# NQAI Voice — VoxCPM2 LoRA Fine-Tune (Colab)

**Amaç:** VoxCPM2 üzerinde ilk NQAI LoRA pilotunu çalıştırmak. Başlangıç hedefi NIVA gibi dar-domain, net konuşan bir ses; NEEKO karakter sesi bu pipeline kanıtlandıktan sonra gelir.

**Model:** `openbmb/VoxCPM2` — Apache-2.0, resmi Türkçe destekli, LoRA/full fine-tune destekli.

**Donanım:** VoxCPM2 LoRA için resmi doküman yaklaşık 20 GB VRAM tahmini verir. T4 16 GB riskli; L4/A100/4090 sınıfı GPU daha rahat.

**Beklenen veri:** JSONL manifest + WAV kayıtlar. Her satır: `{"audio": "path/to/clip.wav", "text": "birebir transcript"}`. Clip uzunluğu pratik tatlı nokta 3-30 saniye.

**Önemli:** ElevenLabs veya izinsiz üçüncü kişi sesleri training data olarak kullanılmaz. Bu notebook sadece bize ait / izinli kayıtlarla çalıştırılır.

## 0. Çalışma Planı

1. GPU ve paketleri doğrula.
2. VoxCPM reposunu clone et, training dependency'lerini kur.
3. Drive çalışma klasörünü hazırla.
4. Manifest ve ses dosyalarını validate et.
5. Train / val / test split üret.
6. LoRA YAML config yaz.
7. Eğitimi başlat.
8. TensorBoard ve checkpoint inference ile kaliteyi dinle.
9. Çıktıları zip'le ve Drive'a bırak.

## 1. GPU Kontrolü

In [ ]:
!python -V
!nvidia-smi

import torch

assert torch.cuda.is_available(), "GPU yok. Colab Runtime -> Change runtime type -> GPU seç."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} · VRAM: {vram_gb:.1f} GB")

if vram_gb < 18:
    print("UYARI: VoxCPM2 LoRA için bu GPU sınırda. batch_size/max_batch_tokens düşürülecek; OOM olabilir.")

## 2. Kurulum — VoxCPM + Training Dependencies

Bu hücre VoxCPM GitHub reposunu `/content/VoxCPM` altına clone eder. Training script'i resmi repo içindeki `scripts/train_voxcpm_finetune.py` dosyasından çalışır.

In [ ]:
import os
import subprocess
from pathlib import Path

def run(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

run("pip install -q --upgrade pip")
run("pip install -q voxcpm soundfile librosa tensorboard tensorboardX argbind safetensors transformers accelerate huggingface_hub pyyaml")

vox_repo = Path("/content/VoxCPM")
if vox_repo.exists():
    run("rm -rf /content/VoxCPM")
run("git clone --depth 1 https://github.com/OpenBMB/VoxCPM.git /content/VoxCPM")

if (vox_repo / "requirements.txt").exists():
    run("pip install -q -r requirements.txt", cwd=str(vox_repo))
run("pip install -q -e .", cwd=str(vox_repo))

print("VoxCPM repo ready:", vox_repo)

## 3. Drive Çalışma Klasörü

Beklenen yapı:

```text
MyDrive/nqai-voice/finetune/niva-pilot-v0/
  manifest_raw.jsonl
  audio/
    0001.wav
    0002.wav
    ...
```

`manifest_raw.jsonl` içinde `audio` alanı mutlak path olabilir veya `audio/` klasörüne göre relatif dosya adı olabilir.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

VOICE_ID = "niva-pilot-v0"
PROJECT_DIR = Path(f"/content/drive/MyDrive/nqai-voice/finetune/{VOICE_ID}")
AUDIO_DIR = PROJECT_DIR / "audio"
RAW_MANIFEST = PROJECT_DIR / "manifest_raw.jsonl"
SPLIT_DIR = PROJECT_DIR / "splits"
CONF_DIR = PROJECT_DIR / "conf"
CKPT_DIR = PROJECT_DIR / "checkpoints" / "lora"
LOG_DIR = PROJECT_DIR / "logs" / "lora"
OUT_DIR = PROJECT_DIR / "outputs"
MODEL_DIR = Path("/content/drive/MyDrive/nqai-voice/models/VoxCPM2")

for p in [PROJECT_DIR, AUDIO_DIR, SPLIT_DIR, CONF_DIR, CKPT_DIR, LOG_DIR, OUT_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_MANIFEST:", RAW_MANIFEST)
print("AUDIO_DIR:", AUDIO_DIR)

## 4. Model Ağırlıklarını Drive'a İndir

İlk çalıştırmada uzun sürebilir. Drive'a indiği için sonraki Colab oturumlarında tekrar indirme azalır.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="openbmb/VoxCPM2",
    local_dir=str(MODEL_DIR),
    local_dir_use_symlinks=False,
)
print("MODEL_DIR:", MODEL_DIR)
print("Files:", sorted(p.name for p in MODEL_DIR.iterdir())[:20])

## 5. Manifest Validate

Bu hücre eğitim başlamadan önce veri hatalarını yakalar: eksik dosya, boş transcript, çok kısa/uzun clip, okunamayan audio.

In [ ]:
import json
import random
from collections import Counter

import soundfile as sf

MIN_SECONDS = 1.0
IDEAL_MIN_SECONDS = 3.0
MAX_SECONDS = 30.0

assert RAW_MANIFEST.exists(), f"Manifest bulunamadı: {RAW_MANIFEST}"

def resolve_audio_path(raw_audio):
    p = Path(raw_audio)
    if p.is_absolute():
        return p
    if (PROJECT_DIR / p).exists():
        return PROJECT_DIR / p
    return AUDIO_DIR / p

records = []
warnings = []
with RAW_MANIFEST.open("r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        item = json.loads(line)
        audio_path = resolve_audio_path(item["audio"])
        text = str(item.get("text", "")).strip()
        if not audio_path.exists():
            raise FileNotFoundError(f"Line {line_no}: audio yok: {audio_path}")
        if not text:
            raise ValueError(f"Line {line_no}: text boş")
        info = sf.info(str(audio_path))
        duration = float(info.frames / info.samplerate)
        if duration < MIN_SECONDS:
            raise ValueError(f"Line {line_no}: çok kısa ({duration:.2f}s): {audio_path}")
        if duration > MAX_SECONDS:
            raise ValueError(f"Line {line_no}: çok uzun ({duration:.2f}s): {audio_path}")
        if duration < IDEAL_MIN_SECONDS:
            warnings.append(f"Line {line_no}: idealden kısa ({duration:.2f}s): {audio_path.name}")
        records.append({
            "audio": str(audio_path),
            "text": text,
            "duration": round(duration, 3),
            "speaker_id": item.get("speaker_id", VOICE_ID),
        })

assert records, "Manifest boş."
durations = [r["duration"] for r in records]
print(f"OK: {len(records)} clip · toplam {sum(durations)/60:.1f} dk · ortalama {sum(durations)/len(durations):.2f}s")
print("Speaker dağılımı:", Counter(r["speaker_id"] for r in records))
if warnings[:10]:
    print("\nUyarılar ilk 10:")
    for w in warnings[:10]:
        print("-", w)

## 6. Train / Val / Test Split

Train satırlarının yaklaşık %40'ına aynı speaker'dan `ref_audio` eklenir. VoxCPM dokümanı, modelin hem LoRA hem referans tabanlı cloning yeteneğini koruması için training samples'ın %30-50'sinde `ref_audio` kullanılmasını öneriyor.

In [ ]:
SEED = 20260524
VAL_RATIO = 0.10
TEST_RATIO = 0.10
REF_AUDIO_RATIO = 0.40

rng = random.Random(SEED)
records_shuffled = records[:]
rng.shuffle(records_shuffled)

n = len(records_shuffled)
n_test = max(1, int(n * TEST_RATIO)) if n >= 10 else 0
n_val = max(1, int(n * VAL_RATIO)) if n >= 10 else 0

test_records = records_shuffled[:n_test]
val_records = records_shuffled[n_test:n_test + n_val]
train_records = records_shuffled[n_test + n_val:]

by_speaker = {}
for r in train_records:
    by_speaker.setdefault(r["speaker_id"], []).append(r)

def with_ref_audio(split_records, enable_ref=False):
    out = []
    for r in split_records:
        item = {"audio": r["audio"], "text": r["text"], "duration": r["duration"], "dataset_id": 0}
        if enable_ref and rng.random() < REF_AUDIO_RATIO:
            candidates = [c for c in by_speaker.get(r["speaker_id"], []) if c["audio"] != r["audio"]]
            if candidates:
                item["ref_audio"] = rng.choice(candidates)["audio"]
        out.append(item)
    return out

train_jsonl = with_ref_audio(train_records, enable_ref=True)
val_jsonl = with_ref_audio(val_records, enable_ref=False)
test_jsonl = with_ref_audio(test_records, enable_ref=False)

def write_jsonl(path, rows):
    with Path(path).open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

TRAIN_MANIFEST = SPLIT_DIR / "train.jsonl"
VAL_MANIFEST = SPLIT_DIR / "val.jsonl"
TEST_MANIFEST = SPLIT_DIR / "test.jsonl"

write_jsonl(TRAIN_MANIFEST, train_jsonl)
write_jsonl(VAL_MANIFEST, val_jsonl)
write_jsonl(TEST_MANIFEST, test_jsonl)

print("train:", len(train_jsonl), TRAIN_MANIFEST)
print("val:", len(val_jsonl), VAL_MANIFEST)
print("test:", len(test_jsonl), TEST_MANIFEST)
print("train ref_audio ratio:", sum("ref_audio" in r for r in train_jsonl), "/", len(train_jsonl))

## 7. LoRA Config Üret

Varsayılan pilot config: `r=32`, `alpha=32`, `enable_lm=true`, `enable_dit=true`, `enable_proj=false`. Küçük GPU'da batch ve token limiti otomatik düşürülür.

In [ ]:
import yaml

if vram_gb < 20:
    batch_size = 2
    grad_accum_steps = 8
    max_batch_tokens = 3072
elif vram_gb < 30:
    batch_size = 4
    grad_accum_steps = 4
    max_batch_tokens = 4096
else:
    batch_size = 8
    grad_accum_steps = 2
    max_batch_tokens = 8192

train_minutes = sum(r["duration"] for r in train_records) / 60
rough_steps = max(300, min(2000, int(len(train_records) * 3 / max(batch_size, 1))))

config = {
    "pretrained_path": str(MODEL_DIR),
    "train_manifest": str(TRAIN_MANIFEST),
    "val_manifest": str(VAL_MANIFEST),
    "sample_rate": 16000,
    "out_sample_rate": 48000,
    "batch_size": batch_size,
    "grad_accum_steps": grad_accum_steps,
    "num_workers": 2,
    "num_iters": rough_steps,
    "log_interval": 10,
    "valid_interval": max(100, rough_steps // 2),
    "save_interval": max(100, rough_steps // 2),
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "warmup_steps": min(100, max(20, rough_steps // 10)),
    "max_steps": rough_steps,
    "max_batch_tokens": max_batch_tokens,
    "save_path": str(CKPT_DIR),
    "tensorboard": str(LOG_DIR),
    "lambdas": {"loss/diff": 1.0, "loss/stop": 1.0},
    "lora": {
        "enable_lm": True,
        "enable_dit": True,
        "enable_proj": False,
        "r": 32,
        "alpha": 32,
        "dropout": 0.0,
    },
}

CONFIG_PATH = CONF_DIR / "voxcpm2_lora.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True), encoding="utf-8")

print(f"Train minutes: {train_minutes:.1f}")
print("CONFIG_PATH:", CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding="utf-8"))

## 8. Eğitimi Başlat

OOM olursa önce `batch_size`, sonra `max_batch_tokens` düşür. TTS fine-tune'da aşırı uzun eğitim kaliteyi bozabilir; iyi checkpoint'i dinleyerek seç.

In [ ]:
%cd /content/VoxCPM
!python scripts/train_voxcpm_finetune.py --config_path "{CONFIG_PATH}"

## 9. TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{LOG_DIR}"

## 10. Checkpoint Inference — Dinleyerek Seç

Aşağıdaki hücre `latest` LoRA checkpoint'i ile sabit NQAI test cümlelerini üretir. Overfit belirtisi: farklı metinlerde aynı/benzer ses içeriği, metni görmezden gelme, durmama, gürültü.

In [ ]:
import time

import soundfile as sf
from IPython.display import Audio, Markdown, display
from voxcpm import VoxCPM

LORA_CKPT = CKPT_DIR / "latest"
assert LORA_CKPT.exists(), f"LoRA checkpoint yok: {LORA_CKPT}"

model = VoxCPM.from_pretrained(
    str(MODEL_DIR),
    lora_weights_path=str(LORA_CKPT),
    load_denoiser=False,
)

eval_texts = [
    ("01", "Merhaba, size nasıl yardımcı olabilirim?"),
    ("02", "Lütfen birkaç saniye bekleyin, bilgilerinizi kontrol ediyorum."),
    ("03", "Randevunuzu yarın saat on dört otuz olarak güncelliyorum."),
    ("04", "Anladım, yaşadığınız sorun için üzgünüm. Şimdi birlikte çözelim."),
    ("05", "Yüzde elli indirim kampanyanız bugün sona eriyor."),
]

infer_dir = OUT_DIR / "latest_infer"
infer_dir.mkdir(parents=True, exist_ok=True)

for sid, text in eval_texts:
    t0 = time.time()
    wav = model.generate(text=text, cfg_value=2.0, inference_timesteps=10)
    elapsed = time.time() - t0
    out_path = infer_dir / f"{sid}.wav"
    sf.write(str(out_path), wav, model.tts_model.sample_rate)
    display(Markdown(f"**{sid}** · {elapsed:.1f}s · {text}"))
    display(Audio(str(out_path)))

## 11. Base vs LoRA A/B

Bu hücre aynı cümleyi base VoxCPM2 ve LoRA ile üretir. LoRA'nın gerçekten iyileştirip iyileştirmediğini kulakla hızlı test ederiz.

In [ ]:
ab_text = "Merhaba, NIVA hattına hoş geldiniz. Size en hızlı şekilde yardımcı olacağım."

base_model = VoxCPM.from_pretrained(str(MODEL_DIR), load_denoiser=False)
base_wav = base_model.generate(text=ab_text, cfg_value=2.0, inference_timesteps=10)
base_path = OUT_DIR / "ab_base.wav"
sf.write(str(base_path), base_wav, base_model.tts_model.sample_rate)

lora_wav = model.generate(text=ab_text, cfg_value=2.0, inference_timesteps=10)
lora_path = OUT_DIR / "ab_lora.wav"
sf.write(str(lora_path), lora_wav, model.tts_model.sample_rate)

display(Markdown("**Base VoxCPM2**"))
display(Audio(str(base_path)))
display(Markdown("**VoxCPM2 + LoRA**"))
display(Audio(str(lora_path)))

## 12. Artifact Export

In [ ]:
import datetime
import shutil

run_meta = {
    "voice_id": VOICE_ID,
    "model": "openbmb/VoxCPM2",
    "date_utc": datetime.datetime.utcnow().isoformat(),
    "train_manifest": str(TRAIN_MANIFEST),
    "val_manifest": str(VAL_MANIFEST),
    "test_manifest": str(TEST_MANIFEST),
    "config_path": str(CONFIG_PATH),
    "checkpoint_dir": str(CKPT_DIR),
    "output_dir": str(OUT_DIR),
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 2),
    "train_clips": len(train_jsonl),
    "val_clips": len(val_jsonl),
    "test_clips": len(test_jsonl),
}

meta_path = PROJECT_DIR / "run_metadata.json"
meta_path.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

zip_base = PROJECT_DIR / f"{VOICE_ID}-voxcpm2-lora-artifacts"
if Path(str(zip_base) + ".zip").exists():
    Path(str(zip_base) + ".zip").unlink()
shutil.make_archive(str(zip_base), "zip", root_dir=str(PROJECT_DIR), base_dir="outputs")

print("metadata:", meta_path)
print("zip:", str(zip_base) + ".zip")

## Stop / Devam Kararı

İlk pilotten sonra karar tablosu:

| Sinyal | Karar |
| --- | --- |
| LoRA sesi base'e göre daha tutarlı, metni doğru okuyor | 30 dk → 60 dk dataset büyüt |
| Ses benziyor ama metin bozuluyor | daha erken checkpoint seç, step azalt, transcript temizle |
| Durmama / uzun sessizlik | trailing silence temizle, uzun clipleri böl |
| Gürültü / metalik ses | noisy samples çıkar, kayıt zincirini düzelt |
| Base zaten daha iyi | LoRA config/rank/data kalitesi yeniden ele alınır |

Production'a geçiş için tek koşu yetmez. En az `10dk`, `30dk`, `60dk` üçlüsünü aynı eval setinde A/B dinlemeyle karşılaştır.